In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt
from google.colab import drive

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
IMAGE_SIZE = 299

BATCH_SIZE = 185
CHANNELS = 3
EPOCH = 50

In [7]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/SkinClassification/dataset/DATA/train",
    shuffle= True,
    image_size= (IMAGE_SIZE, IMAGE_SIZE),
    batch_size= BATCH_SIZE

)

Found 440 files belonging to 5 classes.


In [8]:
class_names= dataset.class_names
class_names

['Acne', 'Actinic Keratosis', 'Basal Cell Carcinoma', 'Eczemaa', 'Rosacea']

In [9]:
len(dataset)

3

In [10]:
# plt.figure(figsize=(10,8))
# for image_batch, label_batch in dataset.take(1):
#  for i in range (12):
#    ax= plt.subplot(3,4,i+1)
#    plt.imshow( image_batch[i].numpy().astype("uint8"))
#    plt.title(class_names[label_batch[i]])
#    plt.axis("off")

Dari dataset yang ada, dataset akan dibagi menjadi 2 bagian yaitu: Training Dataset dan Validating Dataset. untuk data training diambil 80% dari dataset yang ada, dan untuk data validasi diambil 20% dari dataset.

In [11]:
def get_dataset(ds, train_split = 0.8, val_split= 0.2, shuffle= True, shuffle_size= 10000):

  ds_size = len(ds)

  if shuffle:
    ds= ds.shuffle(shuffle_size, seed=12)

  train_size = int(ds_size*train_split)
  val_size = int(ds_size*val_split)

  train_ds= ds.take(train_size)
  val_ds= ds.skip(train_size)



  return train_ds, val_ds

In [12]:
train_ds, val_ds= get_dataset(dataset)

In [13]:
len(train_ds)

2

In [14]:
len(val_ds)

1

In [15]:
train_ds= train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds= val_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

In [16]:
resize_and_rescale= tf.keras.Sequential([
    layers.experimental.preprocessing.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.experimental.preprocessing.Rescaling(1.0/255)

])

In [17]:
data_augmentation = tf.keras.Sequential([
    layers.experimental.preprocessing.RandomFlip("horizontal_and_vertical"),
    layers.experimental.preprocessing.RandomRotation(0.2)
])

In [ ]:
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = EfficientNetB7(weights='imagenet', include_top=False)
base_model.trainable = False
x = data_augmentation
# x = resize_and_rescale
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(5, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# model.compile(
#     optimizer= 'adam',
#     loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits= False),
#     metrics=['accuracy']
# )

258076736/258076736 [==============================] - 2s 0us/step


In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, None, None, 3)]      0         []                            
                                                                                                  
 rescaling_1 (Rescaling)     (None, None, None, 3)        0         ['input_1[0][0]']             
                                                                                                  
 normalization (Normalizati  (None, None, None, 3)        7         ['rescaling_1[0][0]']         
 on)                                                                                              
                                                                                                  
 rescaling_2 (Rescaling)     (None, None, None, 3)        0         ['normalization[0][0]']   

In [ ]:
model.compile(
    optimizer= 'adam',
    loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits= False),
    metrics=['accuracy']
)

In [ ]:
history= model.fit(
    train_ds,
    epochs= EPOCH,
    batch_size= BATCH_SIZE,
    verbose=1,
    validation_data= val_ds
)

Epoch 1/50
2/2 [==============================] - 630s 304s/step - loss: 1.6314 - accuracy: 0.3000 - val_loss: 0.9843 - val_accuracy: 0.6000
Epoch 2/50
2/2 [==============================] - 501s 288s/step - loss: 1.0252 - accuracy: 0.5946 - val_loss: 0.8719 - val_accuracy: 0.6000
Epoch 3/50
2/2 [==============================] - 512s 291s/step - loss: 0.7851 - accuracy: 0.7000 - val_loss: 0.8564 - val_accuracy: 0.7000
Epoch 4/50


In [ ]:
test_ds= tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/SkinClassification/dataset/DATA/testing",
    shuffle= True,
    image_size= (IMAGE_SIZE, IMAGE_SIZE),
    batch_size= BATCH_SIZE

)

In [ ]:
model.evaluate(test_ds)

In [ ]:
history.history.keys()

In [ ]:
acc= history.history['accuracy']
val_acc= history.history['val_accuracy']

loss= history.history['loss']
val_loss= history.history['val_loss']

In [ ]:
plt.figure(figsize= (10,10))
plt.subplot(1,2,1)
plt.plot(range(EPOCH), acc, label= 'Training Accuracy')
plt.plot(range(EPOCH), val_acc, label= 'Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

In [ ]:
plt.figure(figsize= (10,10))
plt.subplot(1,2,1)
plt.plot(range(EPOCH), loss, label= 'Training Loss')
plt.plot(range(EPOCH), val_loss, label= 'Validation Loss')
plt.legend(loc='lower right')
plt.title('Training and Validation loss')

In [ ]:
for image_batch, label_batch in test_ds.take(1):
  first_image= image_batch[0].numpy().astype('uint8')
  first_label= label_batch[0].numpy()


  print("First image to predict:")
  plt.imshow(first_image)
  print("Actual Label: ", class_names[first_label])
  batch_prediction= model.predict(image_batch)
  print("Predicted Label: ", class_names[np.argmax(batch_prediction[0])])

In [ ]:
def predict(model, img):
  img_array= tf.keras.preprocessing.image.img_to_array(images[i].numpy())
  img_array= tf.expand_dims(img_array, 0)

  predictions= model.predict(img_array)

  predicted_class= class_names[np.argmax(predictions[0])]

  return predicted_class

In [ ]:
history_pred_class= np.array([])
history_actual_class= np.array([])

plt.figure(figsize=(20,120))
for images, labels in test_ds.take(1):
  for i in range(185):
    ax= plt.subplot(37,5, i+1)
    plt.imshow(images[i].numpy().astype("uint8"))

    predicted_class= predict(model, images[i].numpy())
    actual_class= class_names[labels[i]]

    history_pred_class= np.append(history_pred_class, predicted_class)
    history_actual_class= np.append(history_actual_class, actual_class)
    plt.title(f"Actual: {actual_class}, \n Predicted: {predicted_class}.")
    plt.axis("off")

In [ ]:
history_pred_class

In [ ]:
history_actual_class

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

conf_matrix= confusion_matrix(history_actual_class, history_pred_class)
print("Confusion Matrix: ")
print(conf_matrix)

In [ ]:
import pandas as pd
import seaborn as sns

conf_matrix_df = pd.DataFrame(conf_matrix,
                     index = ['ACNE','ACTINIC KERATOSIS','BASAL CELL CARCINOMA','ECZEMAA','ROSACEA'],
                     columns = ['ACNE','ACTINIC KERATOSIS','BASAL CELL CARCINOMA','ECZEMAA','ROSACEA'])

#Plotting the confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(conf_matrix_df, annot=True)
plt.title('Confusion Matrix')
plt.ylabel('Actual Values')
plt.xlabel('Predicted Values')
plt.show()

In [ ]:
import cv2

test_img= cv2.imread("/content/drive/MyDrive/SkinClassification/dataset/test_img3.jpg")
# resize_and_rescale(test_img)
# data_augmentation(test_img)
plt.imshow(test_img.astype("uint8"))

predicted_class= predict(model, test_img)
plt.title(f"Actual: Acne & Eczemaa, \n Predicted: {predicted_class}.")
plt.axis("off")
